In [ ]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
import statsmodels.api as sm

In [ ]:
with open("data/raw/characteristics_with_ind_interaction_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
data.head(15)

In [ ]:
first_15_columns = data.iloc[:, :15]
columns_2 = data.iloc[:, 89:97]
columns_3 = data.iloc[:, 171:275]
columns_4 = data.iloc[:, 867:923]

## 拟合优度R^2

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

## 训练集 验证集划分

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

## 线性回归

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    # 计算残差
    model = LinearRegression(fit_intercept=False)
    model.fit(X_train, y_train)
    residuals = np.abs(model.predict(X_train) - y_train)

    # 计算残差的99.9%分位数
    epsilon = np.percentile(residuals, 99.9)

    # 根据计算得到的epsilon值重新拟合模型
    model = SGDRegressor(fit_intercept=False, loss='huber', epsilon=epsilon)
    model.fit(X_train, y_train)
   
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100

## 惩罚线性回归

In [ ]:
from sklearn.linear_model import ElasticNet  
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    model = ElasticNet(fit_intercept=False, alpha=1.0, l1_ratio=0.5)  
    model.fit(X_train, y_train) 
    y_train_pred = model.predict(X_train) 
    y_test_pred = model.predict(X_test)
    pred_y_list.extend(y_test_pred)  
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    # 计算残差
    model = LinearRegression(fit_intercept=False)
    model.fit(X_train, y_train)
    residuals = np.abs(model.predict(X_train) - y_train)

    # 计算残差的99.9%分位数
    epsilon = np.percentile(residuals, 99.9)

    # 根据计算得到的epsilon值重新拟合模型
    model = SGDRegressor(fit_intercept=False, loss='huber', epsilon=epsilon)
    model.fit(X_train, y_train)
   
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100

## 提升回归树BRT

In [ ]:
#增强回归树
from sklearn.ensemble import GradientBoostingRegressor  
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)  
    model.fit(X_train, y_train) 
    y_test_pred = model.predict(X_test) 
    pred_y_list.extend(y_test_pred)  
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100

## 随机森林

In [ ]:
from sklearn.ensemble import RandomForestRegressor   
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    model = RandomForestRegressor(n_estimators=100, random_state=0)  
    model.fit(X_train, y_train) 
    y_train_pred = model.predict(X_train) 
    y_test_pred = model.predict(X_test) 
    pred_y_list.extend(y_test_pred)  
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100

## 降维PCR

In [ ]:

from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    pca = PCA()
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    lasso = LassoCV(cv=5)
    lasso.fit(X_train_pca, y_train)
    n_components_selected = np.sum(lasso.coef_ != 0)
    X_train_selected = X_train_pca[:, np.where(lasso.coef_ != 0)[0]]
    X_test_selected = X_test_pca[:, np.where(lasso.coef_ != 0)[0]]
    rf = RandomForestRegressor(n_estimators=100, random_state=0)
    rf.fit(X_train_selected, y_train)
    y_train_pred = rf.predict(X_train_selected)
    y_test_pred = rf.predict(X_test_selected)
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list)) * 100

## 神经网络模型

In [ ]:
from tensorflow.keras.models import Sequential  
from tensorflow.keras.layers import Dense  
from tensorflow.keras.optimizers import Adam  
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
#单层神经网络
def relu(x):  
    return np.maximum(0, x)  
  
def relu_grad(x):  
    return (x > 0).astype(np.float32)

def mean_squared_error(y, t):
    return 0.5 * np.sum((y - t) ** 2)

class OneLayerNet:

    def __init__(self, input_size, output_size, weight_init_std=0.01):
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, output_size)
        self.params['b1'] = np.zeros(output_size)

    def predict(self, x):
        W1 = self.params['W1']
        b1 = self.params['b1']
    
        a1 = np.dot(x, W1) + b1
        y = a1
        
        return y
        
    def loss(self, x, t):
        y = self.predict(x)
        return mean_squared_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t) ** 2) 
        
        return error
         
    def gradient(self, x, t):
        W1 = self.params['W1']
        b1 = self.params['b1']
        grads = {}
        
        batch_num = x.shape[0]
        
        # forward
        a1 = np.dot(x, W1) + b1
        y = a1
        
        # backward
        dy = (y - t) / batch_num
        grads['W1'] = np.dot(x.T, dy)
        grads['b1'] = np.sum(dy, axis=0)
        
        return grads

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    network = OneLayerNet(input_size = 94, output_size = 1)  
 
    n_epochs = 100000   
    train_size = X_train.shape[0]
    batch_size = 100
    learning_rate = 0.01  

    patience = 20
    best_model_state = None
    best_loss = float('inf')
    counter = 0

    for epoch in range(n_epochs): 
        for i in range(0, train_size, batch_size):
            xi = X_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]  
        
            grad = network.gradient(xi, yi)
            for key in ('W1', 'b1'):
                network.params[key] -= learning_rate * grad[key]
    
        val_loss = network.accuracy(X_val, y_val)
    
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = network.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 2000 == 0:
            train_loss = network.accuracy(X_train, y_train)
            test_loss = network.accuracy(X_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}") 
        
    for key in ('W1', 'b1'):
        net_work.params[key] = best_model_state[key]
        
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
#双层神经网络
class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 初始化权重
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
    
        a1 = np.dot(x, W1) + b1
        z1 = relu(a1)
        a2 = np.dot(z1, W2) + b2
        y = a2  
        
        return y
        
    def loss(self, x, t):
        y = self.predict(x)
        return mean_squared_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t) ** 2) 
        
        return error
        
    def gradient(self, x, t):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
        grads = {}
        
        batch_num = x.shape[0]
        
        # forward
        a1 = np.dot(x, W1) + b1
        z1 = relu(a1)
        a2 = np.dot(z1, W2) + b2
        y = a2
        
        # backward
        dy = (y - t) / batch_num
        grads['W2'] = np.dot(z1.T, dy)
        grads['b2'] = np.sum(dy, axis=0)
        
        da1 = np.dot(dy, W2.T)
        dz1 = relu_grad(a1) * da1
        grads['W1'] = np.dot(x.T, dz1)
        grads['b1'] = np.sum(dz1, axis=0)

        return grads

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    network = TwoLayerNet(input_size = 94, hidden_size = 32, output_size = 1)  
 
    n_epochs = 100000   
    train_size = x_train.shape[0]
    batch_size = 1000 
    learning_rate = 0.01  

    patience = 20
    best_model_state = None
    best_loss = float('inf')
    counter = 0

    for epoch in range(n_epochs): 
        for i in range(0, train_size, batch_size):
            xi = x_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]  
        
            grad = net_work.gradient(xi, yi)
            for key in ('W1', 'b1', 'W2', 'b2'):
                net_work.params[key] -= learning_rate * grad[key]
    
        val_loss = net_work.accuracy(x_val, y_val)
    
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = net_work.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 2000 == 0:
            train_loss = net_work.accuracy(x_train, y_train)
            test_loss = net_work.accuracy(x_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}") 
        
    for key in ('W1', 'b1', 'W2', 'b2'):
        net_work.params[key] = best_model_state[key]
        
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)